- The first Run will Result in Error
- Click on **Runtime** above and select **Restart session**.
- Run again

- Project: https://github.com/rednote-hilab/dots.tts
- Notebook source: https://github.com/Isi-dev/Google-Colab_Notebooks

In [ ]:
#@title 🎙️ dots.tts-soar Studio

import sys
import os


try:
    from dots_tts.runtime import DotsTtsRuntime
    print("✅ dots.tts is already installed and loaded.")
except ImportError:
    print("⏳ Resolving environment safely (Preventing NumPy/CUDA collisions)...")
    !pip install -q "numpy<2.0.0"
    !pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121
    !pip install -q git+https://github.com/rednote-hilab/dots.tts.git gradio
    from dots_tts.runtime import DotsTtsRuntime
    print("✅ Installation complete!")

import torch
import torchaudio
from google.colab import files
from IPython.display import Audio, display
import gradio as gr

# ==========================================
# 2. MODEL INITIALIZATION
# ==========================================
if 'model' not in globals():
    print("🚀 Loading dots.tts-soar model into VRAM...")
    model = DotsTtsRuntime.from_pretrained("rednote-hilab/dots.tts-soar")
    print("✅ Model loaded successfully!")

# ==========================================
# 3. COLAB FORM INTERFACE CONFIGURATION
# ==========================================
#@markdown ---
#@markdown ### 🎛️ Interface Mode Selection
ui_mode = "Colab Native Form" #@param ["Colab Native Form", "Gradio Web UI"]

#@markdown ### 📝 Text Inputs
text_to_read = "Now, this moment, is all we have. The universe can end us at anytime. Humility and gratitude, they help us navigate with tranquility the unpredictable currents of life, and open our eyes to the beauty of this thorny gift." #@param {type:"string"}

#@markdown ### 🔊 Voice Cloning (Native Upload)
#@markdown Check this box to trigger the Colab file uploader for your reference audio when using Native Form mode.
upload_reference_audio = False #@param {type:"boolean"}
prompt_text = "" #@param {type:"string"}

#@markdown ### ⚙️ Advanced Generation Settings
num_sampling_steps = 10 #@param {type:"slider", min:10, max:32, step:1}
guidance_scale = 4.5 #@param {type:"number"}
display_native_player = True #@param {type:"boolean"}
trigger_auto_download = False #@param {type:"boolean"}
#@markdown ---

# ==========================================
# 4. INFERENCE ENGINE (FIXED DICTIONARY UNPACKING)
# ==========================================
def run_inference(text, prompt_aud, prompt_txt, steps, cfg, display_player, download_file):
    if not text.strip():
        print("❌ Error: Target script text cannot be empty.")
        return None

    kwargs = {
        "text": text,
        "num_steps": int(steps),
        "guidance_scale": float(cfg)
    }

    if prompt_aud and os.path.exists(prompt_aud):
        kwargs["prompt_audio_path"] = prompt_aud
        if prompt_txt.strip():
            kwargs["prompt_text"] = prompt_txt

    try:
        print("🎙️ Synthesizing speech...")
        # FIX: The API returns a dictionary, not a tuple.
        result = model.generate(**kwargs)

        waveform = result["audio"]
        sample_rate = result["sample_rate"]

        if torch.is_tensor(waveform):
            waveform = waveform.squeeze().cpu()

        output_filename = "/content/soar_output.wav"
        torchaudio.save(output_filename, waveform.unsqueeze(0), sample_rate)
        print(f"✨ Generation Complete! Saved to: {output_filename}")

        if display_player:
            display(Audio(output_filename, rate=sample_rate))

        if download_file:
            print("Initiating local file download browser trigger...")
            files.download(output_filename)

        return (sample_rate, waveform.numpy()) if ui_mode == "Gradio Web UI" else None

    except Exception as e:
        print(f"❌ Execution Error: {str(e)}")
        return None

# ==========================================
# 5. EXECUTION MATRIX (Upload Handling)
# ==========================================
if ui_mode == "Colab Native Form":
    active_prompt_audio = ""

    if upload_reference_audio:
        print("\n📥 Please upload your reference audio file (WAV/MP3):")
        uploaded_files = files.upload()
        if uploaded_files:
            filename = list(uploaded_files.keys())[0]
            active_prompt_audio = f"/content/{filename}"
            print(f"✅ Successfully loaded {filename} for voice cloning.")
        else:
            print("⚠️ No file uploaded. Proceeding with the default base voice.")

    run_inference(
        text=text_to_read,
        prompt_aud=active_prompt_audio,
        prompt_txt=prompt_text,
        steps=num_sampling_steps,
        cfg=guidance_scale,
        display_player=display_native_player,
        download_file=trigger_auto_download
    )

else:
    def gradio_wrapper(text, p_aud, p_txt, show_player, dl_file):
        res = run_inference(text, p_aud, p_txt, num_sampling_steps, guidance_scale, show_player, dl_file)
        # FIX: Pass the saved file path directly to Gradio instead of a sliced tuple
        return "/content/soar_output.wav" if res else None

    with gr.Blocks(theme=gr.themes.Soft()) as demo:
        gr.Markdown("### dots.tts-soar Studio Dashboard")
        with gr.Row():
            with gr.Column():
                t_input = gr.Textbox(label="Text to Read", value=text_to_read, lines=3)
                p_aud_input = gr.Audio(label="Prompt Audio (Upload)", type="filepath")
                p_txt_input = gr.Textbox(label="Prompt Text (Optional)", value=prompt_text)
                sp_chk = gr.Checkbox(label="Display Player Natively", value=display_native_player)
                dl_chk = gr.Checkbox(label="Trigger Auto-Download", value=trigger_auto_download)
                btn = gr.Button("Synthesize", variant="primary")
            with gr.Column():
                audio_out = gr.Audio(label="Gradio Playback Output", interactive=False)
        btn.click(gradio_wrapper, [t_input, p_aud_input, p_txt_input, sp_chk, dl_chk], audio_out)
    demo.launch(share=True, inline=True)